<a href="https://colab.research.google.com/github/mayankch5678/YOLO-vs-RT-Detr/blob/main/YOLO_Model_und_Deep_SORT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install --upgrade ultralytics deep-sort-realtime opencv-python-headless gdown pyyaml pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 121.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,

In [ ]:
!yolo train model=yolo11s.pt data=/content/drive/MyDrive/Comparison/data1.yaml epochs=50 imgsz=480

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Comparison/data1.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4

In [ ]:
import os
import argparse
import cv2
import numpy as np
import torch
from ultralytics import YOLO, RTDETR

"""
TO RUN THIS SCRIPT

python /home/s124z/Code/YOLO/eval_new.py --weights /home/s124z/Code/YOLO/best.pt --skip_eval --source /home/s124z/E130-Projekte/Saliq_datasets/100clips/Clip/Prokto_5_clip_0697.mp4 --conf_thresh 0.25
Loading trained weights from: /home/s124z/Code/YOLO/best.pt

"""
try:
    from deep_sort_realtime.deepsort_tracker import DeepSort
except ImportError:
    print("DeepSort not found. Please install: pip install deep-sort-realtime")
    DeepSort = None

def parse_args():
    parser = argparse.ArgumentParser(description="Evaluation and Tracking Script")

    # Model Loading
    parser.add_argument('--weights', type=str, required=True, help='/home/s124z/Desktop/E130-Projekte/Saliq_datasets/Final_yolo_ckpts_VideoSplit/yolo.pt')

    # Evaluation Args
    parser.add_argument('--dataset_config', type=str, default='config/dataset/endoscapes.yaml', help='Path to data YAML for validation')
    parser.add_argument('--skip_eval', action='store_true', help='Skip metric calculation and go straight to tracking')

    # Tracking Args
    parser.add_argument('--source', type=str, default='video.mp4', help='/home/s124z/E130-Projekte/Saliq_datasets/100clips/Clip/Rektum_8_clip_1962.mp4')
    parser.add_argument('--conf_thresh', type=float, default=0.5, help='Confidence threshold')

    return parser.parse_args()

def run_tracker(model, source_path, conf_thresh=0.7):
    """
    Runs the Object Detection and Tracking pipeline.
    """
    if DeepSort is None:
        print("Cannot run tracking: DeepSort library missing.")
        return

    print(f"--- Starting Tracking on {source_path} ---")

    # Initialize DeepSort
    tracker = DeepSort(max_age=30, n_init=3)

    # Open video source
    if source_path == '0':
        cap = cv2.VideoCapture(0)
    else:
        cap = cv2.VideoCapture(source_path)

    if not cap.isOpened():
        print(f"Error: Could not open video source {source_path}")
        return

    # Video Properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Output setup
    output_path = os.path.splitext(source_path)[0] + '_tracked.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Object Detection
        results = model(frame, verbose=False, conf=conf_thresh)

        # Prepare detections for DeepSort
        detects = []
        for result in results:
            boxes = result.boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                confidence = float(box.conf[0].cpu().numpy())
                cls_id = int(box.cls[0].cpu().numpy())

                w = x2 - x1
                h = y2 - y1
                detects.append([[x1, y1, w, h], confidence, cls_id])

        # 2. Update Tracker
        tracks = tracker.update_tracks(detects, frame=frame)

        # 3. Draw Tracks
        for track in tracks:
            if not track.is_confirmed():
                continue

            track_id = track.track_id
            ltrb = track.to_ltrb()

            cv2.rectangle(frame, (int(ltrb[0]), int(ltrb[1])), (int(ltrb[2]), int(ltrb[3])), (0, 255, 0), 2)
            cv2.putText(frame, f"ID: {track_id}", (int(ltrb[0]), int(ltrb[1]) - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        out.write(frame)

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Tracking finished. Output saved to {output_path}")

def main():
    args = parse_args()

    # Load Model
    print(f"Loading trained weights from: {args.weights}")

    # Auto-detect model type based on filename usually works,
    # but we can try/except or infer from string
    if 'yolo' in args.weights.lower():
        model = YOLO(args.weights)
    else:
        # Assuming RTDETR if not YOLO
        model = RTDETR(args.weights)

    # --- Evaluation Phase ---
    if not args.skip_eval:
        print("--- Starting Evaluation ---")
        metrics = model.val(data=args.dataset_config)

        # Calculate F1
        precision = np.mean(metrics.box.p)
        recall = np.mean(metrics.box.r)

        if (precision + recall) > 0:
            f1_score = 2 * (precision * recall) / (precision + recall)
        else:
            f1_score = 0.0

        print("\n--- Final Evaluation Metrics ---")
        print(f"Precision (P): {precision:.4f}")
        print(f"Recall (R): {recall:.4f}")
        print(f"F1 Score: {f1_score:.4f}")
        print("--------------------------------")

    # --- Tracking Phase ---
    run_tracker(model, args.source, args.conf_thresh)

if __name__ == "__main__":
    main()